# ForecastIQ — Corrected N-HiTS and DeepAR benchmark

This notebook preserves the earlier DL experiment as an exploratory baseline in the timestamped notebook backup. The corrected workflow uses the fixed 400-series evaluation panel, validation-only configuration selection, the same `d_1886–d_1913` explored benchmark, explicit predictive-mean output, and standardized prediction records.

Set `MODE='fast'` for a resource-conscious smoke test or `MODE='research'` for 750-step, three-origin validation. A fast run is **not** full scientific validation. The research settings are deliberately memory-conscious for a 4-GB GPU.

In [1]:
from pathlib import Path
import sys,json,time,platform
import numpy as np,pandas as pd,torch,neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS,DeepAR
from neuralforecast.losses.pytorch import DistributionLoss
PROJECT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
from src.evaluation.evaluation_panel import validate_panel,prediction_records
from src.experiments.config import ExperimentConfig,save_manifest,sha256
from src.evaluation.nested_selection import (ValidationCache,eligible_inner_origins,
    select_configurations,stable_config_id)
from src.forecasting.neural_models import (cuda_diagnostics,neural_candidates,
    save_training_metadata,training_mode)

OUT=PROJECT/'data'/'processed';CONFIG=ExperimentConfig();SEED=42;H=28
panel=pd.read_parquet(OUT/'evaluation_panel.parquet');validate_panel(panel)
days=[f'd_{i}' for i in range(1,1914)]
calendar=pd.read_csv(PROJECT/'data'/'raw'/'calendar.csv',usecols=['d','date']).drop_duplicates('d').set_index('d')
dates=pd.DatetimeIndex(pd.to_datetime(calendar.reindex(days).date));assert not dates.isna().any()
diagnostics=cuda_diagnostics();display(pd.Series(diagnostics,name='value'))
print('NeuralForecast:',neuralforecast.__version__)
DEVICE='gpu' if diagnostics['cuda_available'] else 'cpu';print('Selected device:',DEVICE)
if not diagnostics['cuda_available'] and diagnostics['torch_cuda_build'] is None:
    print('Cause: this interpreter has a CPU-only PyTorch build. Do not reinstall inside this notebook.')

,value
python,3.13.16
executable,/usr/bin/python3
platform,Linux-6.6.159+-x86_64-with-glibc2.39
torch_version,2.11.0+cu130
torch_cuda_build,13.0
cuda_available,True
device_count,1
device,cuda
device_name,Tesla T4
device_memory_bytes,15637086208


NeuralForecast: 3.3.0
Selected device: gpu


## Validation-only search

Both families use identical chronological origins. DeepAR compares Student-t and Negative Binomial likelihoods on validation only. NeuralForecast's alias column is the Monte-Carlo predictive **mean**; `-median` is retained separately for diagnostics. Negative Binomial is tested, not presumed superior.

In [2]:
MODE='research'  # change to 'research' for meaningful training
RUN_MULTI_ORIGIN=True  # research-only; expensive, and never implied by a single-origin run
mode=training_mode(MODE)
MAX_STEPS=mode.max_steps;VALIDATION_ORIGINS=list(mode.validation_origins)
VAL_CHECK_STEPS=mode.val_check_steps;BATCH_SIZE=mode.batch_size
INPUT_SIZE=mode.input_size;LEARNING_RATE=mode.learning_rate
candidates=neural_candidates(MODE)
for spec in candidates:spec['config_id']=stable_config_id(spec)
CACHE=ValidationCache(PROJECT/'results'/'validation_cache'/'neural')
PANEL_HASH=sha256(OUT/'evaluation_panel.parquet')

def long_frame(origin):
    ids=panel.series_id.astype(str).to_numpy();Y=panel[days[:origin]].to_numpy(np.float32)
    return pd.DataFrame({'unique_id':np.repeat(ids,origin),'ds':np.tile(dates[:origin],len(ids)),'y':Y.reshape(-1)})

def make_model(spec):
    common=dict(h=H,input_size=INPUT_SIZE,max_steps=MAX_STEPS,learning_rate=spec.get('learning_rate',LEARNING_RATE),
                batch_size=BATCH_SIZE,windows_batch_size=mode.windows_batch_size,random_seed=SEED,scaler_type=spec['scaler'],alias=spec['name'],
                early_stop_patience_steps=mode.early_stop_patience_steps,val_check_steps=VAL_CHECK_STEPS,
                accelerator=DEVICE,devices=1,enable_checkpointing=True,default_root_dir=str(PROJECT/'results'/'checkpoints'/spec['name']))
    if spec['family']=='NHITS':
        if spec.get('mlp_units') is not None:common['mlp_units']=spec['mlp_units']
        return NHITS(**common)
    loss=DistributionLoss(distribution=spec['distribution'],num_samples=mode.num_samples)
    return DeepAR(**common,lstm_n_layers=1,lstm_hidden_size=spec['hidden_size'],lstm_dropout=0.,trajectory_samples=100,loss=loss)

def validate_candidates(inner_origins,outer_origin):
    rows=[]
    for spec in candidates:
        for inner in inner_origins:
            key={'panel_hash':PANEL_HASH,'mode':MODE,'outer_origin':outer_origin,'inner_origin':inner,
                 'config_id':spec['config_id'],'seed':SEED,'max_steps':MAX_STEPS}
            cached=CACHE.load(key)
            if cached is not None:rows.append(cached['result']);continue
            start=time.time()
            base={**spec,'outer_origin':outer_origin,'inner_origin':inner,'seed':SEED,
                  'training_cutoff':inner,'evaluation_population':len(panel)}
            try:
                nf=NeuralForecast(models=[make_model(spec)],freq='D');nf.fit(df=long_frame(inner),val_size=H)
                pred=nf.predict().reset_index();actual=panel[[f'd_{i}' for i in range(inner+1,inner+H+1)]].to_numpy(float).reshape(-1)
                point=pred[spec['name']].to_numpy(float)
                if point.shape!=actual.shape or not np.isfinite(point).all():raise ValueError('invalid forecast shape or nonfinite prediction')
                result={**base,'MAE':float(np.abs(point-actual).mean()),'bias':float((point-actual).mean()),
                        'WAPE':float(np.abs(point-actual).sum()/actual.sum()) if actual.sum()>0 else np.nan,
                        'negative_fraction':float((point<0).mean()),'prediction_mean':float(point.mean()),
                        'actual_mean':float(actual.mean()),'point_forecast':'predictive_sample_mean' if spec['family']=='DeepAR' else 'point',
                        'seconds':time.time()-start,'status':'completed','device':DEVICE}
            except Exception as error:
                result={**base,'MAE':np.nan,'bias':np.nan,'seconds':time.time()-start,
                        'status':'failed','error':repr(error),'device':DEVICE}
            CACHE.save(key,result);rows.append(result)
    return pd.DataFrame(rows)

validation=validate_candidates(VALIDATION_ORIGINS,1885)
selected_summary,validation_leaderboard=select_configurations(validation,expected_origins=VALIDATION_ORIGINS)
display(validation_leaderboard);display(selected_summary)
validation.to_csv(OUT/'phase3d_validation_search.csv',index=False)
validation_leaderboard.to_csv(OUT/'phase3d_validation_leaderboard.csv',index=False)
selected_summary.to_csv(OUT/'phase3d_selected_configurations.csv',index=False)

,family,config_id,mean_MAE,std_MAE,n_folds,expected_folds,complete,inner_origins
0,DeepAR,6f9945cec62ef193,3.314377,0.029284,3,3,True,"[1801, 1829, 1857]"
1,DeepAR,fe8704e1c891757e,3.769143,0.061065,3,3,True,"[1801, 1829, 1857]"
2,NHITS,99198bb3998e6c89,3.841848,0.128198,3,3,True,"[1801, 1829, 1857]"
3,NHITS,cab4649eaa10bd2e,3.897524,0.115350,3,3,True,"[1801, 1829, 1857]"


,family,config_id,mean_MAE,std_MAE,n_folds,expected_folds,complete,inner_origins,selection_criterion
0,DeepAR,6f9945cec62ef193,3.314377,0.029284,3,3,True,"[1801, 1829, 1857]",minimum mean MAE across all eligible inner folds
1,NHITS,99198bb3998e6c89,3.841848,0.128198,3,3,True,"[1801, 1829, 1857]",minimum mean MAE across all eligible inner folds


## Refit validation winners and forecast the explored final period

The final period has been inspected previously and is not described as pristine. Configuration choices above use validation only.

In [3]:
selected=selected_summary.to_dict('records');all_records=[];training=[]
for chosen in selected:
    spec=next(c for c in candidates if c['config_id']==chosen['config_id']);start=time.time()
    nf=NeuralForecast(models=[make_model(spec)],freq='D');nf.fit(df=long_frame(1885),val_size=28)
    pred=nf.predict().reset_index();point=pred[spec['name']].to_numpy(float).reshape(len(panel),H)
    records=prediction_records(panel,point,spec['family'],'seen','phase3d-corrected',1885)
    records['point_forecast']='predictive_sample_mean' if spec['family']=='DeepAR' else 'point'
    records['configuration']=spec['name'];all_records.append(records)
    elapsed=time.time()-start
    checkpoint_dir=PROJECT/'results'/'checkpoints'/spec['name']
    checkpoints=[str(p.relative_to(PROJECT)) for p in checkpoint_dir.rglob('*.ckpt')]
    outcome={'model':spec['family'],'configuration':spec['name'],'seconds':elapsed,'device':DEVICE,
             'steps_requested':MAX_STEPS,'checkpoint_files':checkpoints,
             'training_loss':'recorded by Lightning when logger is configured; not fabricated here',
             'early_stopping':'enabled; inspect checkpoint callback/logs'}
    training.append(outcome)
    save_training_metadata(PROJECT/'results'/'training_diagnostics'/f"{spec['name']}_{MODE}.json",
                           mode=mode,candidate=spec,diagnostics=diagnostics,outcome=outcome)
standardized=pd.concat(all_records,ignore_index=True);standardized.to_csv(OUT/'phase3d_standardized_predictions.csv',index=False)
pd.DataFrame(training).to_csv(OUT/'phase3d_training_times.csv',index=False)
diagnostic=(standardized.assign(error=lambda x:x.prediction-x.actual,absolute_error=lambda x:(x.prediction-x.actual).abs())
            .groupby(['model','demand_type'],as_index=False).agg(MAE=('absolute_error','mean'),bias=('error','mean')))
diagnostic.to_csv(OUT/'phase3d_demand_type_diagnostics.csv',index=False);display(diagnostic)
manifest=save_manifest(PROJECT/'results'/'manifests'/'phase3d_corrected.json','phase3d-corrected',CONFIG,
    sources=[OUT/'evaluation_panel.parquet'],extra={'mode':MODE,'device':DEVICE,'candidates':candidates,'validation_origins':VALIDATION_ORIGINS,
    'run_multi_origin':RUN_MULTI_ORIGIN,'selection_metric':'mean validation MAE'})
print('Saved',len(standardized),'matched prediction rows');print('Manifest:',manifest['experiment_id'])

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ DistributionLoss │      5 │ train │     0 │
│ 1 │ valid_loss          │ MAE              │      0 │ train │     0 │
│ 2 │ hist_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 3 │ futr_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 4 │ stat_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 5 │ padder_train        │ ConstantPad1d    │      0 │ train │     0 │
│ 6 │ scaler              │ TemporalNorm     │      0 │ train │     0 │
│ 7 │ hist_encoder        │ LSTM             │  4.5 K │ train │     0 │
│ 8 │ decoder             │ MLP              │     66 │ train │     0 │
└───┴─────────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 5                                                                                            
Total params: 4.6 K                                                                                                
Total estimated model params size (MB): 0.018                                                                      
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

## Optional matched deep-learning multi-origin backtest

This is compute-gated. Every outer origin runs its own inner validation using only target windows ending by that outer origin. Cached results are keyed by panel, configuration, seed, mode, and both origins. It writes selection history and a separate prediction artifact; partial work is never upgraded to a complete benchmark.

In [4]:
if RUN_MULTI_ORIGIN:
    if MODE!='research':raise ValueError('Multi-origin scientific results require MODE=research')
    multi=[];nested_validation=[];selection_history=[]
    for outer in CONFIG.backtest_origins:
        inner=eligible_inner_origins(outer,horizon=H,n_folds=3)
        outer_validation=validate_candidates(inner,outer);nested_validation.append(outer_validation)
        outer_selected,outer_board=select_configurations(outer_validation,expected_origins=inner)
        for row in outer_selected.to_dict('records'):
            spec=next(c for c in candidates if c['config_id']==row['config_id'])
            selection_history.append({**row,'outer_origin':outer,'inner_origins':list(inner),
                'training_cutoff':outer,'seed':SEED,'evaluation_population':len(panel)})
            start=time.time();nf=NeuralForecast(models=[make_model(spec)],freq='D');nf.fit(df=long_frame(outer),val_size=H)
            raw=nf.predict().reset_index()[spec['name']].to_numpy(float).reshape(len(panel),H)
            multi.append(prediction_records(panel,raw,spec['family'],'seen','dl-research-nested',outer))
    dl_multi=pd.concat(multi,ignore_index=True)
    dl_multi.to_csv(OUT/'phase3d_multi_origin_predictions.csv',index=False)
    pd.concat(nested_validation,ignore_index=True).to_csv(OUT/'phase3d_nested_validation_history.csv',index=False)
    pd.DataFrame(selection_history).to_csv(OUT/'phase3d_nested_selection_history.csv',index=False)
    print('Saved',len(dl_multi),'DL multi-origin rows')
else:
    print('DL multi-origin not executed; phase3d_multi_origin_predictions.csv was not modified.')

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ DistributionLoss │      5 │ train │     0 │
│ 1 │ valid_loss          │ MAE              │      0 │ train │     0 │
│ 2 │ hist_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 3 │ futr_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 4 │ stat_cat_embeddings │ ModuleList       │      0 │ train │     0 │
│ 5 │ padder_train        │ ConstantPad1d    │      0 │ train │     0 │
│ 6 │ scaler              │ TemporalNorm     │      0 │ train │     0 │
│ 7 │ hist_encoder        │ LSTM             │  4.5 K │ train │     0 │
│ 8 │ decoder             │ MLP              │     66 │ train │     0 │
└───┴─────────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 5                                                                                            
Total params: 4.6 K                                                                                                
Total estimated model params size (MB): 0.018                                                                      
Modules in train mode: 11                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

In [5]:
from pathlib import Path

export_dir = Path("/content/ForecastIQ_3d_export")
export_dir.mkdir(parents=True, exist_ok=True)

nf.save(path=str(export_dir / "models"), overwrite=True)

print("Trained models saved!")

Trained models saved!
